# 🖼️ SketchToScene — Kaggle Training Notebook

Full training run on Kaggle T4 GPU (16GB VRAM).

**Steps:**
1. Clone repo & install deps
2. Download QuickDraw dataset
3. Train VQ-VAE
4. Train main transformer
5. Push to HuggingFace


In [ ]:
# Step 1: Clone & install
!git clone https://github.com/Sagnik120/sketch2scene.git
%cd sketch2scene
!pip install -e . -q
!pip install einops transformers huggingface_hub wandb -q


In [ ]:
# Step 2: Download data
!python scripts/download_data.py --dataset quickdraw --output data/raw/ --max_per_class 10000


In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# Step 3: Train VQ-VAE separately first
import torch
import sys
sys.path.insert(0, '.')
from src.model.vqvae import VQVAE
from src.data.dataset import build_dataloaders
from torch.optim import AdamW
from tqdm import tqdm

device = 'cuda'
vqvae = VQVAE(base_channels=128, latent_dim=256, n_codes=8192).to(device)
train_dl, val_dl = build_dataloaders('data/raw/quickdraw', img_size=256, batch_size=32, num_workers=2)
opt = AdamW(vqvae.parameters(), lr=1e-4)

vqvae.train()
for epoch in range(10):
    total = 0
    for batch in tqdm(train_dl, desc=f'VQ-VAE epoch {epoch+1}'):
        x = batch['sketch'].to(device)
        out = vqvae(x)
        out['loss'].backward()
        opt.step(); opt.zero_grad()
        total += out['loss'].item()
    print(f'Epoch {epoch+1} | loss={total/len(train_dl):.4f}')

import os; os.makedirs('checkpoints/vqvae', exist_ok=True)
torch.save(vqvae.state_dict(), 'checkpoints/vqvae/vqvae.pt')
print('VQ-VAE saved!')


In [ ]:
# Step 4: Train full SketchToScene model
!python scripts/train.py \
    --config configs/train_config.yaml \
    --device cuda \
    --batch_size 32 \
    --epochs 30 \
    --data_root data/raw/quickdraw \
    --save_dir checkpoints


In [ ]:
# Step 5: Push to HuggingFace
import os
HF_TOKEN = 'hf_YOUR_TOKEN_HERE'  # replace with your token
os.environ['HF_TOKEN'] = HF_TOKEN
!python scripts/export_hf.py --repo Sagnik120/sketch2scene --token $HF_TOKEN


In [ ]:
# Step 6: Quick inference demo
from src.model.sketch2scene import SketchToScene
from src.model.transformer import TransformerConfig
from src.inference.visualize import load_sketch, visualize_results

model = SketchToScene.load('checkpoints/best', TransformerConfig(), device='cuda')

# Use one of the downloaded QuickDraw samples as a sketch
from pathlib import Path
sample = next(Path('data/raw/quickdraw/sketches').glob('*.png'))
sketch = load_sketch(sample, img_size=256).to('cuda')
description = model.generate_description(sketch)
print(f'Generated: {description}')

result = visualize_results(sketch.cpu(), description)
result.save('demo_output.png')
print('Saved demo_output.png')
